<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Numeric Formats

Pure Python/numpy helpers for unsigned integer words and fixed-point raw values.

In [1]:
from fpga_verification.formats import QFormat, UIntFormat

### UIntFormat

You can use `UIntFormat` as custom sized unsigned integer (like standard uint8/uint16)

In [2]:
mono10 = UIntFormat(width=10)
raw_pixels = mono10.array([0, 1023, 1024, -1])

print(raw_pixels.tolist())
print(raw_pixels.dtype)
print(hex(mono10.mask))

[0, 1023, 0, 1023]
uint16
0x3ff


### QFormat

Use `QFormat` to represent fixed-point values. It supports configurable integer (`qi`) and fractional (`qf`) widths. For signed formats, `qi` includes the sign bit, and values are stored using two’s-complement representation.

In [3]:
# 2'b11
# it can be 3 or -1

# unsigned version:
# bin: 00 01 10 11
# dec:  0  1  2  3
q = QFormat(qi=2, qf=0, signed=False)
print( f"0b11 is {q.qraw_to_int(0b11)}" )

# signed version:
# bin: 00 01 10 11
# dec:  0  1  -2  -1
q = QFormat(qi=2, qf=0, signed=True)
print( f"0b10 is {q.qraw_to_int(0b10)}" )


0b11 is 3
0b10 is -2


Use these methods to convert between numeric values, scaled integers,
and raw fixed-point words:

- `float_to_qraw`: encode numeric values as raw words
- `qraw_to_float`: decode raw words as numeric values
- `int_to_qraw`: convert scaled signed integers to raw words
- `qraw_to_int`: convert raw words to scaled signed integers

In [4]:
q = QFormat(qi=3, qf=4, signed=False)
print(f"q.int_to_qraw(5) = {q.int_to_qraw(5)}")
print(f"q.float_to_qraw(5) = {q.float_to_qraw(5)}")


q.int_to_qraw(5) = 5
q.float_to_qraw(5) = 80


Get min/max QFormat range values

In [5]:
q = QFormat(qi=3, qf=4, signed=True)

print(f"min_int: {q.min_int},\t\tmax_int: {q.max_int}")
print(f"min_float: {q.min_float},\tmax_float: {q.max_float}")


min_int: -64,		max_int: 63
min_float: -4.0,	max_float: 3.9375


In [6]:
q = QFormat(qi=3, qf=4, signed=False)

print("Get representation of 3.75 in given QFormat")
print(f"q.float_to_qraw(3.75) = {q.float_to_qraw(3.75)}")
print("Convert back:")
print(f"q.qraw_to_float(15) = {q.qraw_to_float(60)}")

Get representation of 3.75 in given QFormat
q.float_to_qraw(3.75) = 60
Convert back:
q.qraw_to_float(15) = 3.75


Same for arrays

In [7]:
q = QFormat(qi=3, qf=2, signed=True)
raw = q.float_to_qraw([1.25, -1.0, 7.75])
back = q.qraw_to_float(raw)

print(raw.tolist())
print(back.tolist())

[5, 28, 15]
[1.25, -1.0, 3.75]


You can simply get randomized array of this type

In [8]:
q.randomize(10)

array([ 3, 28,  4, 21, 15,  1,  7, 27, 18, 25], dtype=uint8)

Multiplication

In [9]:
# _ _ _ | _ _
q = QFormat(qi=3, qf=2, signed=False)
# _ _ | _ _ _ _
u = QFormat(qi=2, qf=4, signed=False)

# note: 0d10 = 0b1010

# left  = 0 1 0 | 1 0      ~ 2.5
# right =   0 0 | 1 0 1 0  ~ 0.625
res = q.multiply(10, u, 10)

result_format = QFormat(qi=3+2, qf=2+4, signed=False)
result_format.qraw_to_float(res)

np.float64(1.5625)